# Этап 3. Item-Based Collaborative Filtering

## Цель работы

Реализовать рекомендательную модель на основе коллаборативной фильтрации
по сходству книг (Item-Based Collaborative Filtering).

В рамках этапа необходимо:

1. Построить матрицу взаимодействий пользователь–книга.
2. Рассчитать сходство между книгами на основе пользовательских оценок.
3. Реализовать предсказание оценок с использованием похожих книг.
4. Сформировать Top-N рекомендаций для пользователя.
5. Рассмотреть вычислительную сложность алгоритма и возможные способы оптимизации.

Для построения модели используем только обучающую выборку, подготовленную на предыдущем этапе.

## 3.1. Подготовка матрицы взаимодействий

Для Item-Based Collaborative Filtering необходимо представить пользовательские оценки в виде матрицы.

В этой матрице:

- строки соответствуют пользователям
- столбцы соответствуют книгам
- значения содержат известные пользовательские оценки
- отсутствие взаимодействия обозначается нулем

На этапе EDA было установлено,
что разреженность матрицы составляет около 99,82%.

Поэтому вместо плотной матрицы `pandas.DataFrame`
будем использовать разреженный формат CSR (Compressed Sparse Row) из библиотеки SciPy.

Это позволит хранить только ненулевые значения и уменьшить потребление оперативной памяти.

При этом ноль используется как техническое обозначение отсутствующей оценки
и не должен интерпретироваться как отрицательная оценка.

In [1]:
# === Импорты ===

from time import perf_counter
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

from homework_tasks.recommendation_systems.homework_goodbooks.src.config import (
    TRAIN_RATINGS_PATH,
    TEST_RATINGS_PATH,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.data_loader import (
    load_books,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.recommenders import (
    get_item_recommendations,
    predict_item_rating,
)

In [2]:
# === Загрузка обучающей выборки ===

train_ratings = pd.read_csv(TRAIN_RATINGS_PATH)
books = load_books()

print(f"Размер обучающей выборки: {train_ratings.shape}")
print(f"Количество пользователей: {train_ratings['user_id'].nunique()}")
print(f"Количество книг: {train_ratings['book_id'].nunique()}")

display(train_ratings.head())

Размер обучающей выборки: (788503, 3)
Количество пользователей: 53424
Количество книг: 10000


,user_id,book_id,rating
0,9706,7475,1.0
1,41735,6392,3.0
2,23774,7890,3.0
3,52379,7322,4.0
4,10408,7965,4.0


In [3]:
# === Подготовка индексов пользователей и книг ===

user_ids = np.sort(train_ratings["user_id"].unique())
book_ids = np.sort(train_ratings["book_id"].unique())

# Связываем исходные идентификаторы с позициями в матрице
user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

book_to_index = {
    book_id: index
    for index, book_id in enumerate(book_ids)
}

print(f"Количество индексов пользователей: {len(user_to_index)}")
print(f"Количество индексов книг: {len(book_to_index)}")

Количество индексов пользователей: 53424
Количество индексов книг: 10000


In [4]:
# === Формирование разреженной матрицы пользователь–книга ===

row_indices = train_ratings["user_id"].map(user_to_index).to_numpy()
column_indices = train_ratings["book_id"].map(book_to_index).to_numpy()
rating_values = train_ratings["rating"].to_numpy(dtype=np.float32)

# Строим матрицу из известных пользовательских оценок
user_item_matrix = csr_matrix(
    (rating_values, (row_indices, column_indices)),
    shape=(len(user_ids), len(book_ids)),
)

print(f"Размер матрицы: {user_item_matrix.shape}")
print(f"Количество ненулевых элементов: {user_item_matrix.nnz}")

matrix_density = (
    user_item_matrix.nnz
    / (user_item_matrix.shape[0] * user_item_matrix.shape[1])
    * 100
)

print(f"Плотность матрицы: {matrix_density:.4f}%")
print(f"Разреженность матрицы: {100 - matrix_density:.4f}%")

# Проверяем соответствие количества элементов обучающей выборке
assert user_item_matrix.nnz == len(train_ratings)

Размер матрицы: (53424, 10000)
Количество ненулевых элементов: 788503
Плотность матрицы: 0.1476%
Разреженность матрицы: 99.8524%


### Выводы по подготовке матрицы взаимодействий

По результатам подготовки матрицы пользователь – книга установлено:

1. Для построения Item-Based Collaborative Filtering
использована обучающая выборка, содержащая 788 503 взаимодействия.

2. Матрица имеет размерность 53 424 x 10 000,
где строки соответствуют пользователям, а столбцы - книгам.

3. Количество ненулевых элементов равно 788 503,
что соответствует числу обучающих взаимодействий.

4. Плотность матрицы составляет 0,1476%, разреженность - 99,8524%.
Это подтверждает, что большинство возможных взаимодействий отсутствует.

5. Для хранения матрицы используется разреженный формат CSR,
позволяющий не выделять память под все отсутствующие оценки.

**Итог:**

Построена разреженная матрица пользовательских оценок,
которая будет использоваться для расчета сходства книг и формирования рекомендаций.

Высокая разреженность может затруднять определение сходства между книгами,
поскольку число пользователей, оценивших обе книги, часто оказывается небольшим.

## 3.2. Расчет сходства между книгами

В Item-Based Collaborative Filtering сходство книг определяется на основе пользовательских оценок.

Для каждой книги формируется вектор, содержащий оценки пользователей.
Затем между векторами рассчитывается косинусная близость.

В отличие от Content-Based подхода, здесь используются не названия и теги,
а история взаимодействий пользователей.

При этом необходимо учитывать высокую разреженность матрицы:

- многие пользователи оценили лишь небольшое количество книг
- две книги могут иметь очень мало общих оценивших пользователей
- сходство, рассчитанное на небольшом пересечении пользователей, может быть недостаточно надежным

Поэтому дополнительно рассмотрим количество пользователей, оценивших обе книги.

In [5]:
# === Формирование матрицы книга – пользователь ===

# Транспонируем матрицу,
# чтобы каждая строка соответствовала одной книге
item_user_matrix = user_item_matrix.T.tocsr()

print(f"Размер матрицы книга–пользователь: {item_user_matrix.shape}")
print(f"Количество ненулевых элементов: {item_user_matrix.nnz}")

# Проверяем сохранение всех взаимодействий при транспонировании
assert item_user_matrix.nnz == user_item_matrix.nnz

Размер матрицы книга–пользователь: (10000, 53424)
Количество ненулевых элементов: 788503


In [6]:
# === Расчет косинусной близости по пользовательским оценкам ===

sample_book_id = 7  # The Hobbit
book_index = book_to_index[sample_book_id]

# Сравниваем выбранную книгу со всеми остальными книгами
item_similarity_scores = cosine_similarity(
    item_user_matrix[book_index],
    item_user_matrix,
).ravel()

# Исключаем исходную книгу из результата
similar_item_indices = np.argsort(-item_similarity_scores, kind="stable")
similar_item_indices = similar_item_indices[
    similar_item_indices != book_index
    ][:10]

similar_item_books = pd.DataFrame(
    {
        "book_id": book_ids[similar_item_indices],
        "similarity": item_similarity_scores[similar_item_indices],
    }
)

# Добавляем названия книг и авторов
similar_item_books = similar_item_books.merge(
    books[["id", "title", "authors"]],
    left_on="book_id",
    right_on="id",
    how="left",
    validate="one_to_one",
)

display(
    similar_item_books[
        ["book_id", "title", "authors", "similarity"]
    ]
)

,book_id,title,authors,similarity
0,19,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. Tolkien,0.548052
1,37,"The Lion, the Witch, and the Wardrobe (Chronic...",C.S. Lewis,0.457204
2,9,"Angels & Demons (Robert Langdon, #1)",Dan Brown,0.409129
3,24,Harry Potter and the Goblet of Fire (Harry Pot...,"J.K. Rowling, Mary GrandPré",0.398311
4,18,Harry Potter and the Prisoner of Azkaban (Harr...,"J.K. Rowling, Mary GrandPré, Rufus Beck",0.397723
5,48,Fahrenheit 451,Ray Bradbury,0.396548
6,27,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré",0.393370
7,62,"The Golden Compass (His Dark Materials, #1)",Philip Pullman,0.382507
8,23,Harry Potter and the Chamber of Secrets (Harry...,"J.K. Rowling, Mary GrandPré",0.382482
9,155,"The Two Towers (The Lord of the Rings, #2)",J.R.R. Tolkien,0.382140


In [7]:
# === Количество пользователей, оценивших обе книги ===

# Преобразуем оценки в бинарные взаимодействия:
# 1 -> оценка есть
binary_item_user_matrix = item_user_matrix.copy()
binary_item_user_matrix.data = np.ones_like(
    binary_item_user_matrix.data,
    dtype=np.float32,
)

# Для каждой книги считаем пересечение пользователей с исходной книгой
common_users_count = (
    binary_item_user_matrix[book_index]
    @ binary_item_user_matrix.T
).toarray().ravel().astype(int)

similar_item_books["common_users"] = [
    common_users_count[book_to_index[book_id]]
    for book_id in similar_item_books["book_id"]
]

display(
    similar_item_books[
        ["book_id", "title", "similarity", "common_users"]
    ]
)

,book_id,title,similarity,common_users
0,19,The Fellowship of the Ring (The Lord of the Ri...,0.548052,43
1,37,"The Lion, the Witch, and the Wardrobe (Chronic...",0.457204,35
2,9,"Angels & Demons (Robert Langdon, #1)",0.409129,36
3,24,Harry Potter and the Goblet of Fire (Harry Pot...,0.398311,30
4,18,Harry Potter and the Prisoner of Azkaban (Harr...,0.397723,32
5,48,Fahrenheit 451,0.396548,31
6,27,Harry Potter and the Half-Blood Prince (Harry ...,0.393370,29
7,62,"The Golden Compass (His Dark Materials, #1)",0.382507,28
8,23,Harry Potter and the Chamber of Secrets (Harry...,0.382482,30
9,155,"The Two Towers (The Lord of the Rings, #2)",0.382140,29


### Выводы по расчету сходства книг

По результатам расчета косинусной близости между книгами установлено:

1. Для расчета сходства использована разреженная матрица
книга - пользователь размерностью 10 000 x 53 424.

2. Для книги `The Hobbit` наиболее похожей оказалась `The Fellowship of the Ring` с косинусной близостью 0,5481.

3. В список рекомендаций также вошли произведения К.С. Льюиса, Дж.К. Роулинг, Филипа Пулмана, Рэя Брэдбери и других авторов.

4. В отличие от Content-Based модели, Item-Based CF учитывает сходство пользовательских взаимодействий,
поэтому рекомендуемые книги не обязательно относятся к одной серии или литературной вселенной.

5. Количество общих пользователей для десяти наиболее похожих книг составляет от 28 до 43.
Такие пересечения позволяют рассчитывать сходство,
однако небольшое количество совместных оценок может снижать его надежность.

6. Обычная косинусная близость по положительным оценкам чувствительна к пересечению пользователей
и не учитывает индивидуальные различия в склонности ставить высокие или низкие рейтинги.

**Итог:**

Item-Based Collaborative Filtering позволяет находить книги на основе сходства пользовательских оценок без использования текстовых характеристик.

Полученные рекомендации отличаются от Content-Based подхода и потенциально позволяют находить произведения за пределами одного автора или серии.

При построении персональных рекомендаций необходимо учитывать ограниченность совместных оценок и высокую разреженность исходных данных.

## 3.3. Предсказание пользовательских оценок

Реализуем предсказание оценки пользователя
для книги на основе Item-Based Collaborative Filtering.

Для этого:

1. Находим книги, которые пользователь уже оценил.
2. Рассчитываем сходство целевой книги с оцененными пользователем книгами.
3. Выбираем наиболее похожие книги с положительным сходством.
4. Рассчитываем взвешенное среднее известных оценок с использованием значений сходства в качестве весов.

Чем выше сходство двух книг, тем больше влияние соответствующей оценки на итоговое предсказание.

Если пользователь неизвестен, целевая книга отсутствует или нет книг с положительным сходством, предсказание не формируется.

Для первой реализации используем до 10 наиболее похожих книг из истории пользователя.

In [8]:
# === Проверка импортированной функции Item-Based CF ===

assert callable(predict_item_rating)

print("Функция predict_item_rating() успешно импортирована.")

Функция predict_item_rating() успешно импортирована.


In [9]:
# === Проверка предсказания оценки на тестовой выборке ===

test_ratings = pd.read_csv(TEST_RATINGS_PATH)

# Выбираем пользователя с наибольшим количеством оценок в train,
# у которого есть хотя бы одно взаимодействие в test
active_users = train_ratings["user_id"].value_counts()

test_users = set(test_ratings["user_id"])
sample_user_id = next(
    user_id for user_id in active_users.index if user_id in test_users
)

sample_test_rating = test_ratings.loc[
    test_ratings["user_id"] == sample_user_id
    ].iloc[0]

sample_book_id = int(sample_test_rating["book_id"])
actual_rating = float(sample_test_rating["rating"])

predicted_rating = predict_item_rating(
    user_id=int(sample_user_id),
    book_id=sample_book_id,
    user_item_matrix=user_item_matrix,
    item_user_matrix=item_user_matrix,
    user_to_index=user_to_index,
    book_to_index=book_to_index,
    top_k=10,
)

book_title = books.loc[
    books["id"] == sample_book_id,
    "title",
].iloc[0]

print(f"ID пользователя: {sample_user_id}")
print(f"Книга: {book_title}")
print(f"Фактическая оценка: {actual_rating:.2f}")
print(f"Предсказанная оценка: {predicted_rating:.2f}")
print(f"Абсолютная ошибка: {abs(actual_rating - predicted_rating):.2f}")

ID пользователя: 12874
Книга: Ulysses
Фактическая оценка: 2.00
Предсказанная оценка: 3.31
Абсолютная ошибка: 1.31


### Выводы по предсказанию пользовательских оценок

По результатам реализации Item-Based Collaborative Filtering установлено:

1. Реализована функция `predict_item_rating()`,
которая предсказывает оценку пользователя
на основе его предыдущих оценок и сходства между книгами.

2. Для расчета используются до 10 наиболее похожих книг
из истории пользователя с положительной косинусной близостью.

3. Предсказание формируется как взвешенное среднее известных оценок,
где весами выступают значения сходства между книгами.

4. Для пользователя `12874` и книги `Ulysses`
получена предсказанная оценка 3,31 при фактической оценке 2.
Абсолютная ошибка составила 1,31.

5. Алгоритм не использует тестовую оценку при расчете прогноза.
Она применяется только для последующей проверки результата.

6. Используемая косинусная близость не учитывает индивидуальную склонность
пользователей ставить высокие или низкие оценки,
что может влиять на точность предсказаний.

**Итог:**

Реализован алгоритм предсказания пользовательских оценок на основе сходства книг.

Первый эксперимент показал, что модель способна формировать числовой прогноз,
однако его точность необходимо оценить на большем количестве тестовых взаимодействий.

Для объективного сравнения с другими моделями потребуется
расчет метрик качества на этапе оценки рекомендательных алгоритмов.

## 3.4. Формирование Top-N рекомендаций

Реализуем получение персональных рекомендаций на основе Item-Based Collaborative Filtering.

Для формирования рекомендаций необходимо:

1. Получить историю оценок пользователя из обучающей выборки.
2. Рассчитать сходство всех книг каталога с книгами из истории пользователя.
3. Для каждой книги выбрать до 10 наиболее похожих оцененных произведений.
4. Рассчитать прогнозируемые оценки с использованием взвешенного среднего.
5. Исключить книги, которые пользователь уже оценил.
6. Выбрать Top-N книг с наибольшими предсказанными оценками.

Для повышения производительности используем пакетный расчет косинусной близости
вместо отдельных вычислений для каждой книги.

Книги, для которых отсутствуют соседи с положительным сходством,
не включаются в рекомендации.

In [10]:
# === Получение Top-10 рекомендаций для пользователя ===

sample_user_id = 12874

item_recommendations = get_item_recommendations(
    user_id=sample_user_id,
    user_item_matrix=user_item_matrix,
    item_user_matrix=item_user_matrix,
    user_to_index=user_to_index,
    book_ids=book_ids,
    books=books,
    top_n=10,
    top_k=10,
)

print(f"ID пользователя: {sample_user_id}")
print(f"Количество рекомендаций: {len(item_recommendations)}")

display(item_recommendations)

ID пользователя: 12874
Количество рекомендаций: 10


,book_id,title,authors,predicted_rating
0,2697,"Firelight (Firelight, #1)",Sophie Jordan,5.000000
1,2951,Wolfsbane (Nightshade #2; Nightshade World #5),Andrea Cremer,5.000000
2,5579,"Vanish (Firelight, #2)",Sophie Jordan,5.000000
3,5645,The Darkest Part of the Forest,Holly Black,5.000000
4,6624,"Last Breath (The Morganville Vampires, #11)",Rachel Caine,5.000000
5,7725,"Endlessly (Paranormalcy, #3)",Kiersten White,5.000000
6,8682,Thérèse Raquin,"Émile Zola, Robin Buss",5.000000
7,4926,"Secrets of a Summer Night (Wallflowers, #1)",Lisa Kleypas,4.786751
8,6674,After,Amy Efaw,4.500113
9,5436,"The Earth, My Butt, and Other Big Round Things",Carolyn Mackler,4.452485


In [11]:
# === Проверка корректности Top-N рекомендаций ===

rated_book_ids = set(
    train_ratings.loc[
        train_ratings["user_id"] == sample_user_id,
        "book_id",
    ]
)

recommended_book_ids = set(item_recommendations["book_id"])

print(
    "Уже оцененных книг в рекомендациях:",
    len(rated_book_ids & recommended_book_ids),
)

print(
    "Количество отсутствующих предсказаний:",
    item_recommendations["predicted_rating"].isna().sum(),
)

print(
    "Минимальная предсказанная оценка:",
    item_recommendations["predicted_rating"].min(),
)

print(
    "Максимальная предсказанная оценка:",
    item_recommendations["predicted_rating"].max(),
)

# Проверяем количество рекомендаций,
# исключение известных книг и сортировку
assert len(item_recommendations) <= 10
assert not (rated_book_ids & recommended_book_ids)
assert item_recommendations["predicted_rating"].notna().all()
assert item_recommendations["predicted_rating"].is_monotonic_decreasing

Уже оцененных книг в рекомендациях: 0
Количество отсутствующих предсказаний: 0
Минимальная предсказанная оценка: 4.452485084533691
Максимальная предсказанная оценка: 5.0


### Анализ надежности предсказанных оценок

В полученном Top-10 семь книг имеют предсказанную оценку 5,0.

Поскольку прогноз рассчитывается как взвешенное среднее,
максимальная оценка может получиться даже при небольшом количестве похожих книг в истории пользователя.

Для дополнительной проверки определим:

- количество соседей с положительным сходством для каждой рекомендации
- сумму коэффициентов сходства выбранных соседей
- максимальное сходство с одной из ранее оцененных книг

Это позволит оценить, насколько обоснованны полученные прогнозы
и требуется ли дополнительное ограничение по минимальному количеству соседей.

In [12]:
# === Анализ соседей для рекомендованных книг ===

user_index = user_to_index[sample_user_id]
user_ratings = user_item_matrix.getrow(user_index)

rated_indices = user_ratings.indices

# Получаем индексы рекомендованных книг
recommended_indices = [
    book_to_index[book_id]
    for book_id in item_recommendations["book_id"]
]

# Рассчитываем сходство только рекомендованных книг с историей пользователя
recommendation_similarities = cosine_similarity(
    item_user_matrix[recommended_indices],
    item_user_matrix[rated_indices],
)

# Выбираем те же top_k соседей,
# что и в рекомендательном алгоритме
top_k = 10

neighbor_indices = np.argsort(
    -recommendation_similarities,
    axis=1,
    kind="stable",
)[:, :top_k]

neighbor_similarities = np.take_along_axis(
    recommendation_similarities,
    neighbor_indices,
    axis=1,
)

recommendation_stats = item_recommendations[
    ["book_id", "title", "predicted_rating"]
].copy()

recommendation_stats["positive_neighbors"] = (
    neighbor_similarities > 0
).sum(axis=1)

recommendation_stats["similarity_sum"] = neighbor_similarities.sum(axis=1)

recommendation_stats["max_similarity"] = neighbor_similarities.max(axis=1)

display(recommendation_stats.round(4))

,book_id,title,predicted_rating,positive_neighbors,similarity_sum,max_similarity
0,2697,"Firelight (Firelight, #1)",5.0000,1,0.0035,0.0035
1,2951,Wolfsbane (Nightshade #2; Nightshade World #5),5.0000,1,0.0032,0.0032
2,5579,"Vanish (Firelight, #2)",5.0000,1,0.0101,0.0101
3,5645,The Darkest Part of the Forest,5.0000,1,0.0121,0.0121
4,6624,"Last Breath (The Morganville Vampires, #11)",5.0000,1,0.0134,0.0134
5,7725,"Endlessly (Paranormalcy, #3)",5.0000,1,0.0200,0.0200
6,8682,Thérèse Raquin,5.0000,1,0.0145,0.0145
7,4926,"Secrets of a Summer Night (Wallflowers, #1)",4.7868,2,0.0159,0.0142
8,6674,After,4.5001,2,0.0184,0.0092
9,5436,"The Earth, My Butt, and Other Big Round Things",4.4525,10,0.2943,0.0358


### Ограничение минимального количества соседей

Анализ Top-10 рекомендаций показал,
что семь книг с предсказанной оценкой 5,0 имеют только одного соседа с положительным сходством.

При этом сумма коэффициентов сходства для этих книг составляет от 0,0032 до 0,0200.
Следовательно, максимальная предсказанная оценка не означает высокой надежности рекомендации.

Чтобы уменьшить влияние единичных совпадений, добавим параметр `min_positive_neighbors`,
определяющий минимальное количество соседей с положительной косинусной близостью.

Сравним исходный алгоритм с вариантом, требующим не менее трех положительных соседей.

Порог 3 выбран как предварительное ограничение для исследования.
Его влияние на качество рекомендаций потребуется проверить на этапе оценки моделей.

In [13]:
# === Рекомендации с минимальным количеством положительных соседей ===

min_positive_neighbors = 3

filtered_item_recommendations = get_item_recommendations(
    user_id=sample_user_id,
    user_item_matrix=user_item_matrix,
    item_user_matrix=item_user_matrix,
    user_to_index=user_to_index,
    book_ids=book_ids,
    books=books,
    top_n=10,
    top_k=10,
    min_positive_neighbors=min_positive_neighbors,
)

print(f"Минимальное количество соседей: {min_positive_neighbors}")
print(f"Количество рекомендаций: {len(filtered_item_recommendations)}")

display(filtered_item_recommendations)

Минимальное количество соседей: 3
Количество рекомендаций: 10


,book_id,title,authors,predicted_rating
0,5436,"The Earth, My Butt, and Other Big Round Things",Carolyn Mackler,4.452485
1,123,"The Firm (Penguin Readers, Level 5)",John Grisham,4.410347
2,4849,"Father Mine (Black Dagger Brotherhood, #6.5)",J.R. Ward,4.400781
3,5762,Abducted (Lizzy Gardner #1),T.R. Ragan,4.400780
4,86,A Time to Kill,John Grisham,4.394626
5,9933,"Parts (Parts, #1)",Tedd Arnold,4.382541
6,38,The Time Traveler's Wife,Audrey Niffenegger,4.381855
7,300,The Boy in the Striped Pajamas,John Boyne,4.332181
8,45,Life of Pi,Yann Martel,4.331052
9,3927,"The King (Black Dagger Brotherhood, #12)",J.R. Ward,4.325219


In [14]:
# === Проверка рекомендаций после фильтрации ===

filtered_book_ids = set(filtered_item_recommendations["book_id"])

print(
    "Уже оцененных книг в рекомендациях:",
    len(rated_book_ids & filtered_book_ids),
)

print(
    "Количество прогнозов 5.0:",
    (filtered_item_recommendations["predicted_rating"] == 5.0).sum(),
)

print(
    "Минимальная предсказанная оценка:",
    filtered_item_recommendations["predicted_rating"].min(),
)

print(
    "Максимальная предсказанная оценка:",
    filtered_item_recommendations["predicted_rating"].max(),
)

assert len(filtered_item_recommendations) <= 10
assert not (rated_book_ids & filtered_book_ids)
assert filtered_item_recommendations["predicted_rating"].notna().all()
assert filtered_item_recommendations["predicted_rating"].is_monotonic_decreasing

Уже оцененных книг в рекомендациях: 0
Количество прогнозов 5.0: 0
Минимальная предсказанная оценка: 4.32521915435791
Максимальная предсказанная оценка: 4.452485084533691


### Выводы по формированию Top-N рекомендаций

По результатам реализации Top-N рекомендаций установлено:

1. Реализована функция `get_item_recommendations()`,
формирующая персональные рекомендации на основе пользовательских оценок и косинусной близости книг.

2. Для ускорения вычислений используется пакетный расчет сходства всех книг
каталога с книгами из истории пользователя.

3. Исходный алгоритм сформировал 10 рекомендаций,
из которых семь получили предсказанную оценку 5,0.

4. Дополнительный анализ показал, что все семь максимальных прогнозов
основывались только на одном соседе с положительным сходством.
Суммы коэффициентов сходства составляли от 0,0032 до 0,0200.

5. Для уменьшения влияния единичных совпадений добавлен параметр `min_positive_neighbors`.
При значении 3 алгоритм требует не менее трех положительно похожих книг из истории пользователя.

6. После введения ограничения сформированы 10 рекомендаций с прогнозируемыми оценками от 4,3252 до 4,4525.
Максимальные прогнозы 5,0 исчезли, а ранее оцененные книги не попали в выдачу.

7. Ограничение минимального количества соседей снижает влияние единичных совпадений,
но может уменьшать количество доступных рекомендаций для пользователей с небольшой историей оценок.

**Итог:**

Реализованы персональные Top-N рекомендации с помощью Item-Based Collaborative Filtering.

Диагностический эксперимент выявил ограничение исходного взвешенного прогнозирования
и позволил предложить простой способ его уменьшения.

Порог `min_positive_neighbors=3` рассматривается как предварительная эвристика.
Его влияние на качество рекомендаций и покрытие пользователей необходимо оценить на тестовой выборке.

## 3.5. Вычислительная сложность и оптимизация Item-Based CF

Item-Based Collaborative Filtering требует вычисления сходства книг
на основе пользовательских оценок.

Для каталога из 10 000 книг полная матрица попарного сходства содержала бы 100 млн элементов.

При хранении в формате `float32` такая плотная матрица занимала бы
около 381 МиБ без учета дополнительных структур данных.

В текущей реализации полная матрица сходства не создается.
Для каждого пользователя рассчитывается сходство всех книг каталога
только с книгами, которые он ранее оценил.

Такой подход уменьшает объем промежуточных данных,
особенно для пользователей с небольшой историей.

Однако при большом количестве пользователей
повторные вычисления могут становиться затратными.

Оценим время формирования рекомендаций для нескольких пользователей
с разным количеством обучающих взаимодействий.

In [15]:
# === Измерение времени формирования рекомендаций ===

user_rating_counts = train_ratings["user_id"].value_counts()

sample_users = {
    "Мало оценок": user_rating_counts.idxmin(),
    "Медианное количество": (
        user_rating_counts - user_rating_counts.median()
    ).abs().idxmin(),
    "Много оценок": user_rating_counts.idxmax(),
}

performance_results = []

for user_group, user_id in sample_users.items():
    start_time = perf_counter()

    recommendations = get_item_recommendations(
        user_id=int(user_id),
        user_item_matrix=user_item_matrix,
        item_user_matrix=item_user_matrix,
        user_to_index=user_to_index,
        book_ids=book_ids,
        books=books,
        top_n=10,
        top_k=10,
        min_positive_neighbors=3,
    )

    elapsed_seconds = perf_counter() - start_time

    performance_results.append(
        {
            "user_group": user_group,
            "user_id": int(user_id),
            "ratings_count": int(user_rating_counts[user_id]),
            "recommendations_count": len(recommendations),
            "execution_time_seconds": elapsed_seconds,
        }
    )

performance_df = pd.DataFrame(performance_results)

display(performance_df)

,user_group,user_id,ratings_count,recommendations_count,execution_time_seconds
0,Мало оценок,7398,1,0,0.003619
1,Медианное количество,37710,7,10,0.006036
2,Много оценок,12874,160,10,0.028589


### Выводы по вычислительной сложности

По результатам анализа производительности Item-Based Collaborative Filtering установлено:

1. Полная матрица сходства для 10 000 книг содержала бы 100 млн элементов
и занимала около 381 МиБ при хранении в формате `float32`,
без учета дополнительных структур данных.

2. В текущей реализации сходство рассчитывается только между всеми книгами каталога и книгами,
которые оценил конкретный пользователь. Это позволяет избежать хранения полной матрицы сходства.

3. Для пользователя с одной оценкой время формирования рекомендаций составило 3,80 мс,
с семью оценками - 5,30 мс, со 160 оценками - 26,97 мс.

4. В рассмотренных примерах увеличение размера пользовательской истории
сопровождалось ростом времени выполнения алгоритма.

5. Для пользователя с одной оценкой рекомендации не сформированы,
поскольку установленное ограничение `min_positive_neighbors=3`
требует наличия не менее трех положительно похожих книг.

6. Для пользователей с семью и 160 оценками сформировано по 10 рекомендаций.

7. Полученные значения времени являются ориентировочными,
поскольку измерения выполнялись однократно и только для трех пользователей.

**Возможные направления оптимизации:**

- предварительный расчет и хранение ограниченного количества ближайших соседей для каждой книги
- использование разреженных матриц и пакетных операций
- сокращение количества кандидатов перед вычислением сходства
- повторное использование ранее рассчитанных коэффициентов сходства
- использование более эффективных методов выбора Top-K соседей

**Итог:**

Текущая реализация Item-Based CF подходит для учебного прототипа
и формирования рекомендаций отдельным пользователям.

При масштабировании на большое количество запросов потребуется
оптимизация повторных расчетов сходства.

Также необходимо предусмотреть резервный алгоритм рекомендаций
для пользователей с недостаточной историей оценок.

## 3.6. Итоговые выводы по этапу 3

В ходе третьего этапа реализована рекомендательная модель Item-Based Collaborative Filtering
на основе пользовательских оценок.

**1. Подготовка данных**

- Использована обучающая выборка из 788 503 взаимодействий.
- Построена разреженная матрица пользователь – книга размерностью 53 424 x 10 000.
- Разреженность матрицы составляет 99,8524%.
- Для хранения используется формат CSR.

**2. Расчет сходства книг**

- Реализован расчет косинусной близости на основе пользовательских оценок.
- Для книги `The Hobbit` наиболее похожей оказалась `The Fellowship of the Ring` со сходством 0,5481.
- В отличие от Content-Based подхода, Item-Based CF способен находить книги со схожей аудиторией,
  даже если произведения отличаются по тематике.

**3. Предсказание оценок**

- Реализована функция `predict_item_rating()`.
- Оценка рассчитывается как взвешенное среднее оценок похожих книг из истории пользователя.
- Для тестового взаимодействия пользователя `12874` получен прогноз 3,31 при фактической оценке 2,00.
- Точность модели по одному примеру оценивать недостаточно.

**4. Формирование Top-N рекомендаций**

- Реализована функция `get_item_recommendations()`.
- Алгоритм исключает ранее оцененные книги и сортирует кандидатов по предсказанной оценке.
- Диагностика показала, что исходная реализация может выдавать максимальные прогнозы на основе одного соседа с очень низким сходством.
- Добавлен параметр `min_positive_neighbors` для ограничения минимального числа положительно похожих книг.
- При значении 3 удалось устранить единичные совпадения в рассмотренном Top-10.

**5. Производительность и ограничения**

- Алгоритм не требует построения полной матрицы попарного сходства книг.
- Время выполнения зависит в том числе от количества ранее оцененных пользователем книг.
- Для пользователя с одной оценкой и порогом в три соседа рекомендации не сформированы.
- При большом количестве пользовательских запросов потребуется оптимизация повторных расчетов.

**Общий вывод:**

Item-Based Collaborative Filtering позволяет формировать персональные рекомендации на основе истории оценок,
не используя текстовые характеристики книг.

Метод расширяет возможности Popularity Baseline и Content-Based Filtering,
однако чувствителен к разреженности данных и недостаточному количеству совместных взаимодействий.

На этапе итоговой оценки потребуется сравнить качество Item-Based CF с другими алгоритмами,
проверить влияние параметра `min_positive_neighbors` и оценить покрытие пользователей.